<a href="https://colab.research.google.com/github/NoorDataAnalyst/flyrank-ML-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane frames content decay as a ranking problem. Instead of simply predicting whether a page will lose traffic ($1$) or not ($0$), the system orders all active content pages for a given client based on their relative urgency of traffic erosion. This directly matches how editorial teams operate—focusing limited human bandwidth on the top-$K$ highest-priority pages that need immediate attention.

In [1]:
# Print task framing declaration
print("ML Task Type: Ranking (Learning-to-Rank)")
print("Objective: Order content pages by decay severity to optimize editorial triage.")

ML Task Type: Ranking (Learning-to-Rank)
Objective: Order content pages by decay severity to optimize editorial triage.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target:** Relative Traffic Loss / Decay Severity Score.  
**Proxy Origin:** Because search engine updates and future traffic loss cannot be directly observed in advance, we define a proxy label based on observed historical performance. We compute the percentage change in organic clicks over a 30-day post-evaluation window versus a 30-day pre-evaluation baseline.  
**Ordinal Labeling:**

0 (Stable/Growing): Traffic change $\ge 0\%$

1 (Moderate Decay): Traffic drop between $-1\%$ and $-20\%$

2 (Severe Decay): Traffic drop $> -20\%$

In [2]:
# Example logic for mapping continuous traffic changes into proxy relevance levels
import pandas as pd
import numpy as np

demo_data = pd.DataFrame({'pct_traffic_change': [0.15, -0.05, -0.28, 0.00, -0.42]})

# Assign proxy relevance score based on defined rules
demo_data['relevance_target'] = np.select(
    [
        demo_data['pct_traffic_change'] >= 0,
        (demo_data['pct_traffic_change'] < 0) & (demo_data['pct_traffic_change'] >= -0.20),
        demo_data['pct_traffic_change'] < -0.20
    ],
    [0, 1, 2],
    default=0
)

print(demo_data)

   pct_traffic_change  relevance_target
0                0.15                 0
1               -0.05                 1
2               -0.28                 2
3                0.00                 0
4               -0.42                 2


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary Metric:** NDCG@K (Normalized Discounted Cumulative Gain at $K=5$ and $K=10$).  
**Why it can be defended:** Standard classification metrics (like Accuracy) ignore position order. NDCG@K specifically penalizes the model if a page experiencing severe traffic loss is ranked lower down the queue, ensuring the most critical content issues appear at the very top of the team's dashboard.  
**Secondary Metric:** Precision@K, measuring what fraction of the top-$K$ recommended pages actually experienced significant traffic decay.

In [3]:
from sklearn.metrics import ndcg_score
import numpy as np

# Example: Defending NDCG@K by showing evaluation of ranked outputs
y_true = np.array([[2, 1, 0, 0, 2]])  # True relevance levels
y_score = np.array([[0.9, 0.8, 0.4, 0.1, 0.05]])  # Predicted urgency scores

score = ndcg_score(y_true, y_score, k=5)
print(f"Sample Baseline NDCG@5 Score: {score:.4f}")

Sample Baseline NDCG@5 Score: 0.9050


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Unit of Analysis**: One row = One Content Page (content_hash_id) per Client (client_hash_id) for a given evaluation period

In [2]:
import duckdb
import getpass

# Enter your Hugging Face token when prompted
HF_TOKEN = getpass.getpass("Paste your Hugging Face READ token: ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'

# Load a real sample slice from DuckDB
df_unit = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        CONCAT(client_hash_id, '_', content_hash_id) AS unit_of_analysis_id,
        SUM(gsc_clicks) AS total_clicks,
        SUM(gsc_impressions) AS total_impressions
    FROM read_parquet('{REL}/fact_content_daily_performance_sample.parquet')
    GROUP BY client_hash_id, content_hash_id
    LIMIT 5
""").df()

print("Real Dataframe Output (Unit of Analysis):")
df_unit

Paste your Hugging Face READ token: ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Real Dataframe Output (Unit of Analysis):


,client_hash_id,content_hash_id,unit_of_analysis_id,total_clicks,total_impressions
0,client_3ffa76342f366962,content_1a6296faee432dae,client_3ffa76342f366962_content_1a6296faee432dae,0.0,0.0
1,client_3ffa76342f366962,content_dc34c661d63e55a9,client_3ffa76342f366962_content_dc34c661d63e55a9,0.0,0.0
2,client_3ffa76342f366962,content_dd83cb75985afc9c,client_3ffa76342f366962_content_dd83cb75985afc9c,0.0,3.0
3,client_3ffa76342f366962,content_42e4dc3c4026a190,client_3ffa76342f366962_content_42e4dc3c4026a190,0.0,0.0
4,client_3ffa76342f366962,content_d86e1c84849226ba,client_3ffa76342f366962_content_d86e1c84849226ba,0.0,0.0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**Multi-Factor Interaction:** Simple if-then rules (e.g., if clicks_drop > 20% then flag) fail because they evaluate metrics in isolation. A traffic drop might be caused by global site seasonality, a broken URL, or search intent shifts.  
**Non-Linear Thresholds:** Machine learning models combine multiple subtle features (content age, historical volatility, impression-to-click ratios, query position distribution) simultaneously to identify true underlying degradation.  
**Dynamic Prioritization:** Fixed rules output flat binary lists, leading to team burnout when hundreds of pages trigger alerts at once. ML ranking dynamicall.y ranks pages so editorial resources are directed to where they have the highest impact.

In [4]:
import pandas as pd
# Code demonstration showing why simple static rules fail on noisy data
sample_pages = pd.DataFrame({
    'content_id': ['P1', 'P2', 'P3'],
    'clicks_drop': [-0.25, -0.25, -0.05],
    'is_seasonal_product': [True, False, False],
    'impression_drop': [0.00, -0.30, -0.40]
})

# Static Rule Flag
sample_pages['rule_flag'] = sample_pages['clicks_drop'] < -0.20

print("Static Rule Output (Fails to separate seasonal noise from structural loss):")
print(sample_pages[['content_id', 'clicks_drop', 'is_seasonal_product', 'rule_flag']])

Static Rule Output (Fails to separate seasonal noise from structural loss):
  content_id  clicks_drop  is_seasonal_product  rule_flag
0         P1        -0.25                 True       True
1         P2        -0.25                False       True
2         P3        -0.05                False      False


## Self-check

Before you submit, confirm each line honestly:

- [✔] Every section above is filled — markdown thinking AND the code that backs it
- [✔] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✔] No client names, URLs, or private queries anywhere
- [✔] My claims use careful words: observed, measured, directional, decision-support
- [✔] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.